# 7.2 经典音源分离方法

用同一段合成混合音频展示 HPSS、NMF 与 REPET，目的在于看清这些方法的假设、中间量和失败边界。


## 1. 环境准备


In [ ]:
import os
import sys
import tempfile
from pathlib import Path

# matplotlib/numba 缓存目录：用跨平台的系统临时目录
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "mplconfig"))
os.environ.setdefault("NUMBA_CACHE_DIR", str(Path(tempfile.gettempdir()) / "numba_cache"))

# 路径推断：从 cwd 向上找含 CODE/chapter07/_common 的目录；NOTEBOOK_DIR 指向 CODE/chapter07/
_p = Path.cwd()
while not (_p / "CODE" / "chapter07" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter07/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
NOTEBOOK_DIR = _p / "CODE" / "chapter07"
CODE_ROOT = NOTEBOOK_DIR.parent
REPO_ROOT = CODE_ROOT.parent
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

FIG_DIR = NOTEBOOK_DIR / "output_figures"
FIG_DIR.mkdir(exist_ok=True)
OUT_AUDIO_DIR = NOTEBOOK_DIR / "output_audio" / "07_2"
OUT_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

print("NOTEBOOK_DIR:", NOTEBOOK_DIR.relative_to(REPO_ROOT))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from chapter07._common.audio_io import load_audio, save_audio
from chapter07._common.external_diagnostics import find_mixture_paths, repo_relative_path
from chapter07._common.metrics import reconstruction_error, si_sdr
from chapter07._common.plotting import (
    FIGURE_SAVE_DPI,
    GRAY_IMAGE_CMAP,
    GRAY_MASK_CMAP,
    LINE_GRAYS,
    plot_matrix_grid,
    plot_stem_spectrogram_grid,
    plot_waveforms,
)
from chapter07._common.spectrogram import amplitude_to_db, stft
from chapter07._common.synthesis import make_synthetic_mixture
from chapter07.classic.hpss import (
    compute_hpss_masks,
    hpss_separate,
    median_filter_spectrogram,
)
from chapter07.classic.nmf import (
    component_frequency_centroids,
    component_masks,
    nmf_decompose,
)
from chapter07.classic.repet import (
    estimate_period_from_similarity,
    repet_masks,
    repet_separate,
    self_similarity_matrix,
)


## 2. 合成混合音频


In [ ]:
sr = 22050
sources = make_synthetic_mixture(sr=sr, duration=4.0, seed=12)
mixture = sources["mixture"]

plot_waveforms(
    {
        "harmonic": sources["harmonic"],
        "bass": sources["bass"],
        "percussive": sources["percussive"],
        "mixture": mixture,
    },
    sr,
)
plt.show()

save_audio(OUT_AUDIO_DIR / "synthetic_mixture.wav", mixture, sr)


## 3. HPSS：横向/纵向中值滤波


In [ ]:
n_fft = 2048
hop_length = 512
mixture_spec = stft(mixture, n_fft=n_fft, hop_length=hop_length)
mag = np.abs(mixture_spec)

harmonic_enhanced, percussive_enhanced = median_filter_spectrogram(
    mag,
    kernel_harmonic=31,
    kernel_percussive=31,
)
harmonic_mask, percussive_mask = compute_hpss_masks(
    mag,
    kernel_harmonic=31,
    kernel_percussive=31,
    margin=1.0,
)

plot_matrix_grid(
    {
        "混合信号幅度谱 (dB)": amplitude_to_db(mag),
        "横向中值滤波 (dB)": amplitude_to_db(harmonic_enhanced),
        "纵向中值滤波 (dB)": amplitude_to_db(percussive_enhanced),
        "谐波掩蔽": harmonic_mask,
    },
    FIG_DIR / "07_2_hpss_median_filters.png",
    cmap=GRAY_IMAGE_CMAP,
    ncols=2,
)
plt.show()


In [ ]:
hpss_estimates = hpss_separate(
    mixture,
    sr=sr,
    n_fft=n_fft,
    hop_length=hop_length,
    kernel_harmonic=31,
    kernel_percussive=31,
)

hpss_target_harmonic = sources["harmonic"] + sources["bass"]
print("HPSS reconstruction error:", f"{reconstruction_error(mixture, hpss_estimates):.3e}")
print("HPSS harmonic SI-SDR:", f"{si_sdr(hpss_target_harmonic, hpss_estimates['harmonic']):.2f} dB")
print("HPSS percussive SI-SDR:", f"{si_sdr(sources['percussive'], hpss_estimates['percussive']):.2f} dB")

for stem, audio in hpss_estimates.items():
    save_audio(OUT_AUDIO_DIR / f"hpss_{stem}.wav", audio, sr)


HPSS 假设谐波成分在时间方向更连续，打击成分在频率方向更宽。这个假设对鼓点和持续音有效，但对快速琶音、强瞬态钢琴、带节奏门限的合成器会变弱。


## 4. NMF：把频谱写成 W @ H


In [ ]:
W, H = nmf_decompose(mag, n_components=6, random_state=0, max_iter=800)
approx_mag = W @ H
nmf_relative_error = np.linalg.norm(mag - approx_mag) / np.linalg.norm(mag)

centroids = component_frequency_centroids(W, sr=sr, n_fft=n_fft)
order = np.argsort(centroids)
low_group = order[:2].tolist()
rest_group = order[2:].tolist()
masks = component_masks(W, H, {"low_components": low_group, "other_components": rest_group})

print("NMF relative magnitude error:", f"{nmf_relative_error:.3f}")
print("component centroids Hz:", np.round(centroids, 1))
print("low-frequency component ids:", low_group)


In [ ]:
freqs = np.linspace(0, sr / 2, W.shape[0])
times = np.arange(H.shape[1]) * hop_length / sr

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
line_styles = ("-", "--", "-.", ":")
for component in order:
    axes[0, 0].plot(
        freqs,
        W[:, component],
        linewidth=1.0,
        color=LINE_GRAYS[int(component) % len(LINE_GRAYS)],
        linestyle=line_styles[int(component) % len(line_styles)],
        label=f"分量 {component}",
    )
axes[0, 0].set_xlim(0, 5000)
axes[0, 0].set_title("W 基谱")
axes[0, 0].set_xlabel("频率 (Hz)")
axes[0, 0].legend(ncol=2, fontsize=8)

activation_image = axes[0, 1].imshow(
    H[order],
    origin="lower",
    aspect="auto",
    extent=[times[0], times[-1], 0, H.shape[0]],
    cmap=GRAY_IMAGE_CMAP,
)
axes[0, 1].set_title("H 激活矩阵")
axes[0, 1].set_xlabel("时间 (s)")
axes[0, 1].set_ylabel("排序后的分量")
fig.colorbar(activation_image, ax=axes[0, 1], fraction=0.046, pad=0.04)

low_image = axes[1, 0].imshow(
    masks["low_components"],
    origin="lower",
    aspect="auto",
    cmap=GRAY_MASK_CMAP,
    vmin=0,
    vmax=1,
)
axes[1, 0].set_title("低频分量软掩蔽")
axes[1, 0].set_xlabel("帧")
axes[1, 0].set_ylabel("频率 bin")
fig.colorbar(low_image, ax=axes[1, 0], fraction=0.046, pad=0.04)

other_image = axes[1, 1].imshow(
    masks["other_components"],
    origin="lower",
    aspect="auto",
    cmap=GRAY_MASK_CMAP,
    vmin=0,
    vmax=1,
)
axes[1, 1].set_title("其他分量软掩蔽")
axes[1, 1].set_xlabel("帧")
axes[1, 1].set_ylabel("频率 bin")
fig.colorbar(other_image, ax=axes[1, 1], fraction=0.046, pad=0.04)

fig.tight_layout()
fig.savefig(FIG_DIR / "07_2_nmf_wh_components.png", bbox_inches="tight", dpi=FIGURE_SAVE_DPI)
plt.show()


NMF 的 component 不是语义 stem。这里的低频 component 可近似看成 bass-like 区域，但它不会自动知道“贝斯”“鼓”“钢琴”这些标签。


## 5. REPET：重复背景与前景残差


In [ ]:
similarity = self_similarity_matrix(mag)
period = estimate_period_from_similarity(
    similarity,
    min_period_frames=10,
    max_period_frames=90,
)
background_mask, foreground_mask, background_mag = repet_masks(
    mag,
    beat_period_frames=period,
)

print("estimated period in frames:", period)
print("estimated period in seconds:", f"{period * hop_length / sr:.2f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
sim_image = axes[0].imshow(similarity, origin="lower", aspect="auto", cmap=GRAY_IMAGE_CMAP)
axes[0].set_title("帧相似度")
axes[0].set_xlabel("帧")
axes[0].set_ylabel("帧")
fig.colorbar(sim_image, ax=axes[0], fraction=0.046, pad=0.04)

bg_image = axes[1].imshow(
    amplitude_to_db(background_mag),
    origin="lower",
    aspect="auto",
    cmap=GRAY_IMAGE_CMAP,
)
axes[1].set_title("重复背景幅度谱 (dB)")
axes[1].set_xlabel("帧")
axes[1].set_ylabel("频率 bin")
fig.colorbar(bg_image, ax=axes[1], fraction=0.046, pad=0.04)

fg_image = axes[2].imshow(
    foreground_mask,
    origin="lower",
    aspect="auto",
    cmap=GRAY_MASK_CMAP,
    vmin=0,
    vmax=1,
)
axes[2].set_title("前景掩蔽")
axes[2].set_xlabel("帧")
axes[2].set_ylabel("频率 bin")
fig.colorbar(fg_image, ax=axes[2], fraction=0.046, pad=0.04)

fig.tight_layout()
fig.savefig(FIG_DIR / "07_2_repet_similarity.png", bbox_inches="tight", dpi=FIGURE_SAVE_DPI)
plt.show()


In [ ]:
repet_estimates = repet_separate(
    mixture,
    sr=sr,
    n_fft=n_fft,
    hop_length=hop_length,
    beat_period_frames=period,
)

print("REPET reconstruction error:", f"{reconstruction_error(mixture, repet_estimates):.3e}")
for stem, audio in repet_estimates.items():
    save_audio(OUT_AUDIO_DIR / f"repet_{stem}.wav", audio, sr)

plot_stem_spectrogram_grid(repet_estimates, sr)
plt.show()


## 6. 可选：配套素材快速预览


In [ ]:
author_root = NOTEBOOK_DIR.parent / "datasets" / "audio_author"
candidate_mixtures = find_mixture_paths(author_root)

if not candidate_mixtures:
    print("No author mixture found; synthetic demo is complete.")
else:
    path = candidate_mixtures[0]
    audio, real_sr = load_audio(path, sr=22050, mono=True, duration=20.0)
    preview = hpss_separate(audio, sr=real_sr, n_fft=2048, hop_length=512)
    out_path = FIG_DIR / "07_2_optional_author_hpss_preview.png"
    plot_stem_spectrogram_grid({"mixture": audio, **preview}, real_sr, out_path)
    plt.show()
    print("previewed:", repo_relative_path(path, REPO_ROOT))


## 7. 小结

- HPSS 的核心是横向/纵向中值滤波与 soft mask。
- NMF 提供可解释的低秩谱分解，但 component 不天然等于乐器 stem。
- REPET 依赖重复背景假设，适合循环伴奏，不适合不断变化的编曲或自由速度音乐。
